# as-strided-noncontig-source — ex6: 2-D sliding-window image patches + visualize

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-noncontig-source`. Running the final beacon cell reports progress against the `Numpy: Applied patterns and advanced` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-noncontig-source`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-noncontig-source"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## strides and non-contiguity — quick refresher

**Stride** = number of *elements* (not bytes) to advance one step along an axis. A contiguous `(H, W)` float tensor has stride `(W, 1)`.

**`torch.as_strided(input, size, stride)`** builds a zero-copy view at the exact (shape, stride) you specify. It bypasses safety checks — overlapping windows, out-of-bounds offsets, the works. Powerful, dangerous, and the foundation of rolling-window tricks, im2col, and stride-based broadcasting hacks.

**`.contiguous()`** materializes a row-major copy if the current strides aren't already row-major. Required before `.view()`; optional but often a perf-vs-memory trade-off otherwise.

### Exercise 6 — 2-D sliding-window image patches + visualize

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Build a zero-copy 2-D sliding-window view over an image with `as_strided`, then visualize each patch as a tile in a matplotlib grid.
> Keywords: sliding-window, as_strided, image-patches, visualization, im2col
> ```

**KCs targeted:** `strides-anatomy`, `as-strided-rolling-window`, `stride-2d-window`

Implement `ex6_image_patches(img, kh, kw)` to return a 4-D view of shape `(num_h, num_w, kh, kw)` containing every contiguous `kh × kw` patch of the 2-D `img` tensor.

Use `torch.as_strided` so the patches are a **zero-copy view** into the original storage — no data duplication. Hint: given `img` of shape `(H, W)` with stride `(sH, sW)`, the output shape is `(H - kh + 1, W - kw + 1, kh, kw)` and the output stride is `(sH, sW, sH, sW)`.

After your test passes, the visualization cell below renders a grid of every patch as a small `imshow` tile — a debugging trick you'll reach for when checking whether your strides line up with what you intended.

In [ ]:
def ex6_image_patches(img: Tensor, kh: int, kw: int) -> Tensor:
    H, W = img.shape
    sH, sW = img.stride()
    out_h, out_w = H - kh + 1, W - kw + 1
    return t.as_strided(
        img,
        size=(out_h, out_w, kh, kw),
        stride=(sH, sW, sH, sW),
    )


<details><summary>Solution</summary>

```python
def ex6_image_patches(img: Tensor, kh: int, kw: int) -> Tensor:
    H, W = img.shape
    sH, sW = img.stride()
    out_h, out_w = H - kh + 1, W - kw + 1
    return t.as_strided(
        img,
        size=(out_h, out_w, kh, kw),
        stride=(sH, sW, sH, sW),
    )
```

**Why the strides repeat.** The first two axes (`out_h`, `out_w`) walk the *top-left corner* of each patch across the image — same step size as moving one row / column in the source. The last two axes (`kh`, `kw`) walk *within* a patch — also one source row / column. So `(sH, sW, sH, sW)` is exactly right.

**Memory cost.** Zero. The output is a view — same storage, just a different `(size, stride)` interpretation. The materialized 4-D tensor would cost `(H-kh+1) × (W-kw+1) × kh × kw` floats, which for a 224×224 image with 7×7 patches is ~2.4M floats vs ~50K in the source. This is why `as_strided` is the secret sauce behind efficient im2col and convolution implementations.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()